In [ ]:
!pip install -U transformers datasets accelerate scikit-learn pandas numpy seqeval evaluate openpyxl

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer)
import evaluate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
teacher_path = "/content/drive/MyDrive/term_paper_masha/predictions_clean_full_aspects.csv"
manual_path = "/content/drive/MyDrive/term_paper_masha/labeled_reviews_with_additional_with_labels_list.csv"

In [ ]:
LABELS = [
    "CARDS_POS", "CARDS_NEG",
    "DIGITAL_POS", "DIGITAL_NEG",
    "SUPPORT_POS", "SUPPORT_NEG",
    "PAYMENTS_POS", "PAYMENTS_NEG",
    "CREDITS_POS", "CREDITS_NEG",
    "FEES_POS", "FEES_NEG",
    "OFFICE_POS", "OFFICE_NEG",
    "ATM_POS", "ATM_NEG",
    "FRAUD_POS", "FRAUD_NEG",
    "INSURANCE_POS", "INSURANCE_NEG",
    "INVESTMENTS_POS", "INVESTMENTS_NEG",
    "PREMIUM_POS", "PREMIUM_NEG",
    "OTHER_POS", "OTHER_NEG"
]

In [ ]:
BIO_LABELS = ["O"]

for label in LABELS:
    BIO_LABELS.append(f"B-{label}")
    BIO_LABELS.append(f"I-{label}")

label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("BIO labels:", len(BIO_LABELS))

In [ ]:
teacher_df = pd.read_csv(teacher_path, sep=";", encoding="utf-8-sig")

print("teacher_df shape:", teacher_df.shape)
print(teacher_df.columns)
teacher_df.head()

In [ ]:
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return None
    if not isinstance(x, str):
        return None
    x = x.strip()
    if x == "":
        return None
    try:
        return json.loads(x)
    except:
        pass
    try:
        return ast.literal_eval(x)
    except:
        pass

    return None

In [ ]:
def extract_aspects_from_any_row(row):
    parsed = safe_parse(row["prediction"])

    if isinstance(parsed, dict) and "aspects" in parsed:
        parsed = parsed["aspects"]

    aspects = []

    if isinstance(parsed, list):
        for item in parsed:
            if isinstance(item, dict):
                label = item.get("label")
                evidence = item.get("evidence", "")

                if label in LABELS:
                    aspects.append({
                        "label": label,
                        "evidence": str(evidence)
                    })

    return aspects

In [ ]:
teacher_df["aspects"] = teacher_df.apply(extract_aspects_from_any_row, axis=1)
teacher_df["labels_list"] = teacher_df["labels_list"].apply(safe_parse)

In [ ]:
def aspects_to_labels_list(aspects):
    labels = []

    for asp in aspects:
        if isinstance(asp, dict):
            label = asp.get("label")
            if label in LABELS:
                labels.append(label)

    return labels

In [ ]:
def find_span_by_evidence(text, evidence):
    if not isinstance(text, str):
        return None
    if not isinstance(evidence, str):
        return None
    text = text.strip()
    evidence = evidence.strip()
    if evidence == "":
        return None
    start = text.find(evidence)
    if start != -1:
        return start, start + len(evidence)
    text_low = text.lower()
    evidence_low = evidence.lower()
    start = text_low.find(evidence_low)
    if start != -1:
        return start, start + len(evidence)

    evidence_norm = re.sub(r"\s+", " ", evidence_low).strip()
    text_norm = re.sub(r"\s+", " ", text_low).strip()
    start_norm = text_norm.find(evidence_norm)
    if start_norm == -1:
        return None

    return None

In [ ]:
teacher_df["aspects"] = teacher_df.apply(extract_aspects_from_any_row, axis=1)
teacher_df["labels_list"] = teacher_df["aspects"].apply(aspects_to_labels_list)
span_rows = []

for _, row in teacher_df.iterrows():
    review_id = row["id"]
    text = str(row["text"])
    aspects = row["aspects"]
    for asp in aspects:
        label = asp.get("label")
        evidence = asp.get("evidence", "")
        if label not in LABELS:
            continue
        span = find_span_by_evidence(text, evidence)
        if span is None:
            continue

        start, end = span
        span_rows.append({
            "id": review_id,
            "text": text,
            "start": start,
            "end": end,
            "label": label,
            "evidence": text[start:end]
        })

spans_df = pd.DataFrame(span_rows)

print("teacher rows:", len(teacher_df))
print("spans found:", len(spans_df))
spans_df.head()

In [ ]:
def build_spans_for_review(group):
    spans = []

    for _, row in group.iterrows():
        spans.append({
            "start": int(row["start"]),
            "end": int(row["end"]),
            "labels": [row["label"]],
            "text": row["evidence"]
        })

    return spans

token_df = (spans_df.groupby(["id", "text"]).apply(lambda g: build_spans_for_review(g)).reset_index(name="spans"))

print("token_df:", token_df.shape)
token_df.head()

In [ ]:
train_val_df, test_df = train_test_split(token_df, test_size=0.15, random_state=42, shuffle=True)
train_df, val_df = train_test_split(train_val_df, test_size=0.15, random_state=42, shuffle=True)

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

In [ ]:
STUDENT_MODEL = "deepvk/RuModernBERT-base"

In [ ]:
tokenizer_student = AutoTokenizer.from_pretrained(STUDENT_MODEL)

MAX_LENGTH = 512

train_ds = Dataset.from_pandas(train_df[["id", "text", "spans"]])
val_ds = Dataset.from_pandas(val_df[["id", "text", "spans"]])
test_ds = Dataset.from_pandas(test_df[["id", "text", "spans"]])

In [ ]:
def align_labels_with_tokens(example):
    text = example["text"]
    spans = example["spans"]
    encoding = tokenizer_student(text, truncation=True,  padding="max_length", max_length=MAX_LENGTH, return_offsets_mapping=True)
    offsets = encoding["offset_mapping"]
    labels = []
    for start_char, end_char in offsets:
        if start_char == end_char:
            labels.append(-100)
            continue

        token_label = "O"

        for span in spans:
            span_start = int(span["start"])
            span_end = int(span["end"])
            span_label = span["labels"][0]
            if start_char >= span_start and end_char <= span_end:
                if start_char == span_start:
                    token_label = f"B-{span_label}"
                else:
                    token_label = f"I-{span_label}"

                break

        labels.append(label2id[token_label])

    encoding.pop("offset_mapping")
    encoding["labels"] = labels

    return encoding


In [ ]:
train_ds = train_ds.map(align_labels_with_tokens)
val_ds = val_ds.map(align_labels_with_tokens)
test_ds = test_ds.map(align_labels_with_tokens)

columns = ["input_ids", "attention_mask", "labels"]

train_ds.set_format("torch", columns=columns)
val_ds.set_format("torch", columns=columns)
test_ds.set_format("torch", columns=columns)

In [ ]:
student_model = AutoModelForTokenClassification.from_pretrained(
    STUDENT_MODEL,
    num_labels=len(BIO_LABELS),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
seqeval = evaluate.load("seqeval")

In [ ]:
def compute_token_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_predictions = []
    true_labels = []
    for pred, lab in zip(predictions, labels):
        current_preds = []
        current_labels = []
        for p, l in zip(pred, lab):
            if l != -100:
                current_preds.append(id2label[int(p)])
                current_labels.append(id2label[int(l)])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./modernbert_token_absa",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer_student = Trainer(
    model=student_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer_student,
    compute_metrics=compute_token_metrics
)

trainer_student.train()

In [ ]:
test_results = trainer_student.evaluate(test_ds)
test_results

In [ ]:
def predict_token_absa(text):
    inputs = tokenizer_student(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True
    )

    offsets = inputs.pop("offset_mapping")[0].tolist()
    inputs = {key: value.to(student_model.device) for key, value in inputs.items()}
    student_model.eval()
    with torch.no_grad():
        outputs = student_model(**inputs)

    pred_ids = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    pred_labels = [id2label[int(i)] for i in pred_ids]
    entities = []
    current_entity = None
    for label, offset in zip(pred_labels, offsets):
        start, end = offset
        if start == end:
            continue
        if label == "O":
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None
            continue
        if "-" not in label:
            continue
        prefix, entity_label = label.split("-", 1)
        if prefix == "B":
            if current_entity is not None:
                entities.append(current_entity)
            current_entity = {"label": entity_label, "start": start,  "end": end}
        elif prefix == "I":
            if current_entity is not None and current_entity["label"] == entity_label:
                current_entity["end"] = end
            else:
                current_entity = {"label": entity_label, "start": start, "end": end}

    if current_entity is not None:
        entities.append(current_entity)
    result = []
    for ent in entities:
        result.append({
            "label": ent["label"],
            "evidence": text[ent["start"]:ent["end"]],
            "start": ent["start"],
            "end": ent["end"]
        })

    return result

In [ ]:
full_pred_rows = []

for i, row in teacher_df.iterrows():
    review_id = row["id"]
    text = str(row["text"])
    prediction = predict_token_absa(text)
    pred_labels_list = [item["label"] for item in prediction]
    full_pred_rows.append({
        "id": review_id,
        "text": text,
        "student_prediction": prediction,
        "pred_labels_list": pred_labels_list
    })
    if i % 100 == 0:
        print("processed:", i)

student_predictions_df = pd.DataFrame(full_pred_rows)
student_predictions_df.to_csv(OUTPUT_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
student_predictions_df.head()

In [ ]:
manual_df = pd.read_csv(manual_path)

print("manual_df shape:", manual_df.shape)
print(manual_df.columns)
manual_df.head()

In [ ]:
def extract_labels_list_from_row(row):
    if "labels_list" in row.index:
        parsed = safe_parse(row["labels_list"])
        if isinstance(parsed, list):
            return [label for label in parsed if label in LABELS]

    aspects = extract_aspects_from_any_row(row)
    return aspects_to_labels_list(aspects)


teacher_eval_df = teacher_df[["id", "text", "labels_list"]].copy()
teacher_eval_df = teacher_eval_df.rename(columns={"labels_list": "teacher_labels_list"})
manual_df["manual_labels_list"] = manual_df.apply(extract_labels_list_from_row, axis=1)
manual_eval_df = manual_df[["id", "manual_labels_list"]].copy()
student_eval_df = student_predictions_df[["id", "text", "pred_labels_list"]].copy()

In [ ]:
def binarize_labels(labels):
    vector = np.zeros(len(LABELS), dtype=int)
    for label in labels:
        if label in LABELS:
            vector[LABELS.index(label)] = 1

    return vector


def exact_match_accuracy(true_lists, pred_lists):
    scores = []
    for true_labels, pred_labels in zip(true_lists, pred_lists):
        scores.append(set(true_labels) == set(pred_labels))

    return float(np.mean(scores))

In [ ]:
def soft_accuracy_no_extra_penalty(true_labels, pred_labels):
    true_set = set(true_labels)
    pred_set = set(pred_labels)
    if len(true_set) == 0:
        return np.nan

    return len(true_set.intersection(pred_set)) / len(true_set)

In [ ]:
def mean_soft_accuracy(true_lists, pred_lists):
    scores = []
    for true_labels, pred_labels in zip(true_lists, pred_lists):
        score = soft_accuracy_no_extra_penalty(true_labels, pred_labels)
        if not np.isnan(score):
            scores.append(score)
    if len(scores) == 0:
        return np.nan

    return float(np.mean(scores))

In [ ]:
def calculate_metrics_table(true_lists, pred_lists):
    y_true = np.array([binarize_labels(x) for x in true_lists])
    y_pred = np.array([binarize_labels(x) for x in pred_lists])

    precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    exact_match = exact_match_accuracy(true_lists, pred_lists)
    soft_acc = mean_soft_accuracy(true_lists, pred_lists)

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Exact Match Accuracy": exact_match,
        "Soft Accuracy": soft_acc
    }

In [ ]:
student_vs_teacher = student_eval_df.merge(teacher_eval_df[["id", "teacher_labels_list"]], on="id", how="inner")
print("student_vs_teacher:", student_vs_teacher.shape)

In [ ]:
student_vs_manual = student_eval_df.merge(manual_eval_df[["id", "manual_labels_list"]], on="id", how="inner")
print("student_vs_manual:", student_vs_manual.shape)

In [ ]:
metrics_teacher = calculate_metrics_table(true_lists=student_vs_teacher["teacher_labels_list"].tolist(), pred_lists=student_vs_teacher["pred_labels_list"].tolist())
metrics_manual = calculate_metrics_table(true_lists=student_vs_manual["manual_labels_list"].tolist(), pred_lists=student_vs_manual["pred_labels_list"].tolist())
metrics_df = pd.DataFrame({"Compared with teacher predictions": metrics_teacher, "Compared with manual labels": metrics_manual})

metrics_df

In [ ]:
with pd.ExcelWriter(OUTPUT_METRICS_PATH) as writer:
    metrics_df.to_excel(writer, sheet_name="metrics")
    student_vs_teacher.to_excel(writer, sheet_name="student_vs_teacher", index=False)
    student_vs_manual.to_excel(writer, sheet_name="student_vs_manual", index=False)

student_predictions_df.to_csv(OUTPUT_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

trainer_student.save_model(OUTPUT_MODEL_DIR)
tokenizer_student.save_pretrained(OUTPUT_MODEL_DIR)

print("Saved predictions to:", OUTPUT_PREDICTIONS_PATH)
print("Saved metrics to:", OUTPUT_METRICS_PATH)
print("Saved model to:", OUTPUT_MODEL_DIR)

In [ ]:
OUTPUT_PREDICTIONS_PATH = "modernbert_predictions_full_dataset.csv"
OUTPUT_METRICS_PATH = "modernbert_metrics.xlsx"
OUTPUT_MODEL_DIR = "./modernbert_token_absa_final"